<span style="font-family:monospace;letter-spacing:.14em;color:#857259">BANA · MODULE 3 · STATISTICAL REASONING FOR BUSINESS</span>

# Session 1: Chance and distributions: is 18.2% a fact or an accident?

**Data:** EV purchase intention (Kaggle Playground Series S6E9, a *synthetic* dataset). `ev_m3_sample.csv` is the cleaned 10,000-row class sample from Module 2. `ev_m3_population.csv` has 50,000 rows that we treat as **everybody**, so we can see what sampling does.
**Time:** 60 minutes. **Course outcome:** CO3.

| Block | Min | What you do |
|---|---|---|
| Hook | 5 | Our sample says 18.2%. Everybody says 17.4%. Who is wrong? |
| Concept | 10 | Sampling variation, standard error, the bell curve (instructor) |
| Guided lab | 30 | Draw thousands of samples; watch the spread shrink |
| Team task | 10 | How many customers must we survey? |
| Exit ticket | 5 | Three quick checks |

**By the end of this session you can:**
- explain why two samples from the same market give different answers
- say how the spread of sample results shrinks as the sample grows
- explain why averages look bell-shaped even when the data is not
- judge whether a gap between a sample and the truth is plausible luck

**New commands this session (max 5):** `.sample(n, random_state=)`, list comprehension, `np.sqrt`, `stats.binom.sf`, `stats.probplot`

**AI-use (AMBER protocol).** You may use an AI tool to look up *syntax*. The reasoning and interpretation must be your own: you will defend them in the viva.

In [0]:
pip install seaborn

In [0]:
pip install statsmodels

In [0]:
# -- Setup: run once ------------------------------------------------------
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import PercentFormatter
warnings.filterwarnings("ignore")

# House style: Nikhil D Jonathan design system (sand ground, terracotta / lagoon / ochre data colours)
C = {"lagoon": "#25707B", "terracotta": "#B5562F", "ochre": "#E0A231", "moss": "#3F6F4B",
     "earth": "#665643", "ink": "#221B15", "sand": "#F8F3EA", "card": "#FFFDF9", "line": "#E3D6C1"}
sns.set_theme(style="whitegrid", rc={
    "figure.facecolor": C["sand"], "axes.facecolor": C["card"], "axes.edgecolor": "#CDBCA3",
    "grid.color": C["line"], "text.color": C["ink"], "axes.labelcolor": C["ink"],
    "xtick.color": "#4A3E31", "ytick.color": "#4A3E31", "axes.spines.top": False,
    "axes.spines.right": False, "figure.dpi": 100, "axes.titleweight": "bold",
    "axes.titlesize": 12, "axes.titlelocation": "left"})
PCT = PercentFormatter(1, decimals=0)                              # axis labels as percentages
SEQ = sns.light_palette(C["terracotta"], as_cmap=True)   # one-hue sequential map for heatmaps

from scipy import stats
from statsmodels.stats.proportion import proportion_confint, proportions_ztest

# -- Load the data -------------------------------------------------------------
# Needs two files: ev_m3_sample.csv (10,000 rows: the cleaned Module 2 sample)
#                  ev_m3_population.csv (50,000 rows: we treat these as "everybody")
def find(name):
    for folder in [".", "data", "../data"]:
        if os.path.exists(os.path.join(folder, name)):
            return os.path.join(folder, name)
    try:
        from google.colab import files
        print("Choose", name); files.upload(); return name
    except ImportError:
        raise FileNotFoundError(f"Put {name} next to this notebook or in a data folder.")

def add_derived(d):
    """The four derived columns built in Session 1 (pre-written so every notebook stands alone)."""
    d = d.copy()
    d["buy"] = (d["Will_Buy_EV"] == "Yes").astype(int)
    d["age_band"] = pd.cut(d["Age"], bins=[24, 34, 44, 54, 69], labels=["25-34", "35-44", "45-54", "55-69"])
    d["income_band"] = pd.qcut(d["Annual_Income_USD"], 4, labels=["Q1 lowest", "Q2", "Q3", "Q4 highest"])
    d["charger_total"] = d["Charging_Stations_Near_Home"] + d["Charging_Stations_Near_Work"]
    return d

df = add_derived(pd.read_csv(find("ev_m3_sample.csv")))        # the class sample
pop = add_derived(pd.read_csv(find("ev_m3_population.csv")))   # "everybody"
print("Sample:", len(df), "rows | Population:", len(pop), "rows")

## Part A. Hook: our sample versus everybody (5 min)
**Predict first:** the class sample said 18.2% Yes. Do you think *everybody* in the 50,000-row file also says 18.2%? Write your guess.

In [0]:
print(f"Class sample (10,000 rows): {df['buy'].mean():.2%} say Yes")
print(f"Everybody    (50,000 rows): {pop['buy'].mean():.2%} say Yes")

**YOUR ANSWER:** The two numbers differ. Is the class sample *wrong*? What other explanation is there?

> _write here_

## Part B. One sample is one roll of the dice (5 min)
`random_state=i` makes the draw repeatable, so we all see the same samples.

In [0]:
for i in range(5):
    s = pop["buy"].sample(100, random_state=i)        # a random sample of 100 people
    print(f"Sample {i + 1} of 100 people: {s.mean():.0%} say Yes")

**YOUR ANSWER:** Same market, five different answers. Which one is 'right'?

> _write here_

## Part C. Build a sampling distribution (10 min)
We draw **1,000 samples** at each of four sizes, and record the Yes rate every time.
**Predict first:** as the sample grows, will the 1,000 results spread out more, less, or the same?

In [0]:
sizes = [25, 100, 500, 2500]
results = {n: [pop["buy"].sample(n, random_state=i).____() for i in range(1000)] for n in sizes}   # TODO: which method gives the Yes rate?

In [0]:
p = pop["buy"].mean()                                    # the truth for "everybody"
fig, axes = plt.subplots(len(sizes), 1, figsize=(7, 7), sharex=True)
for ax, n in zip(axes, sizes):
    sns.histplot(results[n], binwidth=0.01, color=C["lagoon"], ax=ax)
    ax.axvline(p, color=C["terracotta"], lw=1.6)         # line = the truth
    ax.set_ylabel(f"n = {n}", rotation=0, ha="right", va="center"); ax.set_yticks([]); ax.set_xlabel("")
axes[-1].xaxis.set_major_formatter(PCT)
axes[-1].set_xlabel("Share saying Yes in the sample (red line = everybody)")
plt.tight_layout()

**YOUR ANSWER:** Describe what happens to the spread as n grows. Roughly how many times bigger must n get to *halve* the spread? (Compare n = 100 with n = 2500.)

> _write here_

## Part D. Measure the spread: the standard error (5 min)
The spread of sample results is called the **standard error (SE)**. There is a formula for it.

In [0]:
table = pd.DataFrame({
    "n": sizes,
    "spread_seen": [np.std(results[n]) for n in sizes],           # what we measured
    "formula": [np.____(p * (1 - p) / n) for n in sizes],         # TODO: square root, then compare with spread_seen
}).round(4)
table

**YOUR ANSWER:** Do the two columns agree? What does the formula say about surveying 4 times as many people?

> _write here_

## Part E. The bell curve appears for averages (10 min)
Commute has a spike at exactly 5 km (a floor), so the raw data is **not** bell-shaped. What about the *average of 30 commutes*?
**Predict first:** bell-shaped, or spiky like the raw data?

In [0]:
means30 = [pop["Daily_Commute_km"].sample(30, random_state=i).mean() for i in range(1000)]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
sns.histplot(pop["Daily_Commute_km"], binwidth=2, color=C["ochre"], ax=axes[0]); axes[0].set_title("Raw commutes (everybody)")
sns.histplot(means30, binwidth=0.7, color=C["lagoon"], ax=axes[1]); axes[1].set_title("Average of 30 commutes, 1,000 times")
for ax in axes: ax.set_xlabel("km"); ax.set_ylabel("")
plt.tight_layout()

In [0]:
# Q-Q plot: if the dots hug the line, the shape is close to a bell curve
fig, ax = plt.subplots(figsize=(4.2, 3.6))
stats.probplot(means30, plot=ax)
ax.get_lines()[0].set(color=C["lagoon"], markersize=3); ax.get_lines()[1].set(color=C["terracotta"])
ax.set_title("Q-Q plot of the 1,000 averages")
plt.tight_layout()

**YOUR ANSWER:** What changed between the left and right histogram? Why does this matter when a manager looks at an *average*?

> _write here_

## Part F. Is 18.2% plausible luck? (5 min)
If the truth is 17.4%, how far is 18.2% in standard-error units? Then try the **binomial**: phone 20 people, how many say Yes?

In [0]:
se = np.sqrt(p * (1 - p) / len(df))                      # SE for a sample of 10,000
gap = df["buy"].mean() - p
print(f"Gap = {gap:.2%}; standard error = {se:.2%}; gap is {gap / se:.1f} standard errors")
print(f"Chance that 7 or more of 20 phone calls say Yes: {stats.binom.____(6, 20, p):.1%}")   # TODO: sf = "survival function" = P(more than 6). Which method?

**YOUR ANSWER:** About how many standard errors is the gap? Would you call the Module 2 result a mistake, a surprise, or ordinary luck?

> _write here_

## Team task (10 min): how many customers must we survey?
A manager says: *"Survey 100 customers. If 25% or more say Yes, the market is clearly above the 17% we assumed."*
In your pair, use the `results[100]` list from Part C to find **how often a survey of 100 shows 25% or more when the truth is 17.4%**. Then recommend a sample size and justify it in two sentences.

In [0]:
share = (np.array(results[100]) >= ____).mean()      # TODO: the 25% cut-off, written as a proportion
print(f"{share:.1%} of surveys of 100 people show 25% or more, even though the truth is {p:.1%}")

## Exit ticket (5 min)
1. Two surveys of 200 people from the same market give 15% and 21%. Does that mean one is wrong?
2. You want to *halve* the uncertainty of a survey. How much more data do you need?
3. Why can the average of 30 commutes be bell-shaped when commutes themselves are not?